# ICDM 2026 — *When Are Neural Interaction Discoveries Real?* — Reproducibility Pipeline

**Path A (audit, recommended for reviewers): no GPU, ~30 seconds.**

1. Download `gnavar-icdm-reproducibility.tar.gz` from the repository.
2. In Colab, upload it into the session: **Files** (folder icon, left) **> Upload**.
3. **Runtime > Run all.**

The audit cell unpacks the tarball, recomputes every reported number from the committed result files, and prints a `PASS`/`FAIL` line per claim, ending with `42/42 checks passed`. No datasets, no Python install on your machine, no terminal.

Path B (optional, regenerate result files on a GPU) is at the bottom.

## Path A — audit every paper number

In [ ]:
# Unpack the uploaded bundle and locate its root.
import os, glob, tarfile, sys

tars = glob.glob('/content/*reproducibility*.tar.gz') or glob.glob('/content/*.tar.gz') or glob.glob('*.tar.gz')
assert tars, ('Upload gnavar-icdm-reproducibility.tar.gz first: Files panel (left) > Upload, then Run all again.')
TARBALL = tars[0]
EXTRACT_TO = '/content/gnavar_repro' if os.path.isdir('/content') else os.path.expanduser('~/gnavar_repro')
os.makedirs(EXTRACT_TO, exist_ok=True)
with tarfile.open(TARBALL) as t:
    t.extractall(EXTRACT_TO)

# find the folder that contains results/ and src/
ROOT = None
for dp, dn, fn in os.walk(EXTRACT_TO):
    if 'results' in dn and 'src' in dn:
        ROOT = dp; break
assert ROOT, 'Could not find the bundle root (a folder with results/ and src/) in the tarball.'
print('Unpacked bundle root:', ROOT)
!pip install -q numpy pandas >/dev/null 2>&1
print('Dependencies ready.')

In [ ]:
# === Cell 2: recompute and check every paper number ===
import sys
sys.path.insert(0, ROOT + '/src')
from verifier_core import CHECKS

def _cmp(kind, expected, got, tol):
    if kind == 'exact':  return got == expected
    if kind == 'tol':    return abs(float(got) - float(expected)) <= tol
    if kind == 'range':
        lo, hi = expected
        return (lo <= got[0] and got[1] <= hi) if isinstance(got,(tuple,list)) else (lo <= got <= hi)
    raise ValueError(kind)

def _fmt(v):
    if isinstance(v, float): return f'{v:.4f}'
    if isinstance(v, (tuple, list)): return '(' + ', '.join(_fmt(x) for x in v) + ')'
    return str(v)

n_pass = 0; fails = []; sec = None
for entry in CHECKS:
    section, quantity, kind, expected, fn = entry[:5]
    tol = entry[5] if len(entry) > 5 else 0.0
    if section != sec: print(f'\n[Section {section}]'); sec = section
    try:
        got = fn(ROOT); ok = _cmp(kind, expected, got, tol)
    except Exception as e:
        got = f'ERROR: {e}'; ok = False
    print(f"  [{'PASS' if ok else 'FAIL'}] {quantity:<52} paper={_fmt(expected)}  artifact={_fmt(got)}")
    if ok: n_pass += 1
    else: fails.append((section, quantity, expected, got))

print('\n' + '='*70)
print(f'RESULT: {n_pass}/{len(CHECKS)} checks passed.')
if fails:
    print('FAILED:'); [print(f'  [{s}] {q}: paper={_fmt(e)} artifact={_fmt(g)}') for s,q,e,g in fails]
else:
    print('All paper numbers reproduce from the committed artifacts.')

## Path B — regenerate result files from scratch (optional, GPU)

Not needed to verify the paper. Each notebook in the unpacked bundle's `notebooks/` folder reproduces one `results/` subfolder. To re-run one here:

1. Set a GPU runtime: **Runtime > Change runtime type > GPU**.
2. Open a notebook from `notebooks/` (the three synthetic ones need no data; the three real-data ones need their dataset, sources in `data/README.md`).
3. Run it; it writes to `results/<experiment>/`. Then re-run the Path A cell above to re-check.

Re-runs reproduce the exact-checked quantities (recovery counts, parameter counts, rank orderings). Seed- and hardware-sensitive quantities (held-out MSEs, cross-fit margins, seed-agreement fractions) are verified as ranges and may differ slightly; each `results/<folder>/metadata.json` records the original environment.